# L25 · 下山算法 — 梯度下降

**目标**：要让函数变小，就朝梯度的**反方向**走一小步，反复迭代。

`新位置 = 旧位置 − 学习率 × 梯度`

**为什么对 Aurora 重要**：`aurora.train` 里每次调用 `optimizer.step()`，执行的就是这一行更新规则；学习率设错，收敛曲线就会在极值点两侧震荡或长期停滞。

## 本课剧情：一步步走向低处

梯度告诉你当前位置哪个方向向上，朝反方向走一小步就让函数值下降一点。本节用 `f(x) = (x−3)²` 跑通「计算梯度 → 更新 x → 观察 f(x) 下降」这条流水线，再把同样的规则推广到带噪声的直线拟合。

## 1. 最小化 f(x) = (x−3)²，从 x=0 出发

`f(x) = (x−3)²` 的极小值在 `x=3`，解析梯度为 `f'(x) = 2(x−3)`。从 `x=0` 出发，第一步梯度值 `2(0−3) = −6`：负值说明函数在 `x=0` 处向左上坡，朝右走（减去负梯度）才是下山方向。

**负号是下山的关键**：梯度指向函数上升最陡的方向；减去梯度就是朝下坡走。`lr` 控制步长大小：设成 `0.9` 时步子过大，容易越过极值点在两侧震荡；设成 `0.01` 时步子小，收敛稳定但需要更多步。Adam、SGD、RMSProp 等优化器的本质，都是对 `x_new = x - lr * grad` 这一行的变体——加入动量或自适应步长，但核心更新方向不变。

## 实验入口：用数值变化观察函数

三个关键变量：`x`（当前位置）、`grad`（该点梯度值）、`lr`（学习率）。观察每轮迭代后 `x` 怎样靠近 3，以及 `f(x)` 怎样收敛到 0。

In [ ]:
import numpy as np
grad = lambda x: 2*(x-3)      # f'(x)
x = 0.0; lr = 0.1
for step in range(30):
    x = x - lr * grad(x)
print('收敛到 x =', round(x, 4), ' (最小值在 x=3)')

## 动手观察：变化率不是一句口号

先在单个点上算梯度和一步更新，确认数值正确，再进入多步迭代。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

把 `x` 从 −3 遍历到 3，直接打印每个点的函数值和斜率，看清楚梯度在极值点两侧的符号变化。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现一步更新 `gd_step(x, grad_value, lr)`

**推理路线**：
1. `grad_value` 指向函数上升最快的方向；要让函数值下降，就从当前位置减去它：`x - something`。
2. 步长由 `lr` 控制：实际移动量是 `lr * grad_value`，而不是直接减去整个梯度（梯度数值可能很大）。
3. 合并得到：`x_new = x - lr * grad_value`，整个函数只需一行，不需要循环。

**参考输入输出**：`f(x)=(x−3)²`，当前 `x=0`，梯度 `2(0−3)=−6`，`lr=0.1` → 新 `x = 0 − 0.1×(−6) = 0.6`（向 3 靠近）。

<details><summary>点击查看参考实现</summary>

```python
def gd_step(x, grad_value, lr):
    return x - lr * grad_value
```

</details>

### 写代码前，先把变量表补完整

写 `gd_step` 前明确三件事：
- 输入：`x`（当前位置，float）、`grad_value`（该点梯度值）、`lr`（学习率，控制步长大小）
- 关键步骤：`next_x = x - lr * grad_value`
- 返回：`next_x`，执行一步梯度下降后的新位置

In [ ]:
def gd_step(x, grad_value, lr):
    # ✏️ TODO: 返回更新后的 x
    return None  # ← 改这里


In [ ]:
x = 0.0; lr = 0.1
for _ in range(50):
    x = gd_step(x, 2*(x-3), lr)
assert abs(x - 3.0) < 1e-3
print('✅ 通过：你实现了梯度下降的一步，迭代后 x =', round(x,4))

## 3. 实战预告：用梯度下降拟合一条直线 y = 2x + 1（带噪声）

（这一格直接运行看效果，是 Month 2 线性回归的雏形）

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
X = np.linspace(0, 5, 40)
y = 2*X + 1 + rng.normal(0, 0.5, size=X.shape)   # 真实关系 + 噪声
w, b, lr = 0.0, 0.0, 0.01
for _ in range(2000):
    pred = w*X + b
    err = pred - y
    w -= lr * np.mean(err * X) * 2     # 对 MSE 的梯度
    b -= lr * np.mean(err) * 2
print(f'学到 w={w:.2f} (真值2), b={b:.2f} (真值1)')
plt.scatter(X, y, s=12, label='data')
plt.plot(X, w*X+b, 'r', label='fit')
plt.legend(); plt.title('linear regression via gradient descent'); plt.show()

**🔗 Aurora 连接**：本节实现的 `gd_step(x, grad_value, lr)` 对应 `aurora/train.py` 里 `optimizer.step()` 的核心逻辑。把这套「预测 → 算误差 → 沿梯度更新权重」推广到神经网络参数张量，就是 Month 2–6 训练 ASR、音乐生成和 LLM 的同一个内核。

In [ ]:
def f(x):
    return (x - 2)**2 + 1

x = -2.0
lr = 0.2
for step in range(8):
    grad = 2 * (x - 2)
    print(f'step={step} | x={x:6.3f} | f(x)={f(x):6.3f} | grad={grad:6.3f}')
    x = x - lr * grad
print('你刚刚让 x 一步步靠近了最低点 2。')


## 参数实验：lr 大小决定收敛还是震荡

从 `x=0` 出发，对 `f(x)=(x−3)²` 用 `lr=0.9` 和 `lr=0.1` 分别跑 20 步，打印每步的 `x` 值：

- `lr=0.9`：步子过大，`x` 每步越过极值点 3，在两侧来回震荡（数列不收敛）。
- `lr=0.1`：步子适中，`x` 单调逼近 3，20 步内接近收敛。

观察两组数列的差异，亲眼看清学习率调参对收敛行为的决定性影响。

In [ ]:
def f(x):
    return (x - 2)**2 + 1

x = -2.0
lr = 0.2
for step in range(8):
    grad = 2 * (x - 2)
    print(f'step={step} | x={x:6.3f} | f(x)={f(x):6.3f} | grad={grad:6.3f}')
    x = x - lr * grad
print('观察：x 正在靠近最低点 2。')


## 本课收束

现在能调用 `gd_step(x, grad_value, lr)` 逐步更新 `x`，在 `f(x) = (x−3)²` 上验证收敛过程。这对应 `aurora.train` 训练循环里每一步的参数更新。下一节链式法则会说明多层嵌套函数的梯度如何反向传回每一层，那正是 `gd_step` 需要的梯度来源。

In [ ]:
# 小检查：把“变化一点点”写成可以观察的数字
import numpy as np

def f(x):
    return x**2

x = 3.0
for h in [1.0, 0.1, 0.01, 0.001]:
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'h={h:6.3f} -> 近似斜率 {slope:.6f}')
print('理论值 2*x =', 2*x)
